# Uber

---
## Setup

In [1]:
import json
import sqlite3

import pandas as pd
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from IPython.display import display
from pydantic import BaseModel

## Data model and flow

Our data model will consist of drivers, riders and rides.

A rider will use a POST request to create a new ride with 'unconfirmed' status, and will be returned a fare estimate.

In the schema, we will enforce that a rider can only have 1 unconfirmed ride at a time. We return an HTTP 409 when  attempting to create another unconfirmed ride.

The driver's client regularly sends PATCH requests (`update_location`) to update the driver's location.

A POST request (`reserve_ride`) to `/reservations` from the driver will provisionally assign a ride to the driver and return the details. The endpoint is idempotent: if the driver already holds a reservation, it returns the same ride rather than reserving a new one, so retries are safe.

In a real system, we would use optimistic concurrency control to resolve race conditions where multiple drivers may try reserving a single ride.
If there were contention issues across multiple nodes, we would need a paradigm such as 2-phase commit or SAGA pattern. However, we partition based on location, and riders, drivers and rides will be on the same node.

A column in `rides.status` will indicate that the ride is `reserved`. If the driver accepts this ride (which will be a PATCH request `accept_ride`), `rides.status` is set to `confirmed`.

## Schema

In [171]:
SCHEMA = """
CREATE TABLE riders (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL
);

CREATE TABLE drivers (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    location VARCHAR(12),
    reserved_ride_id INTEGER REFERENCES rides(id)
);

CREATE TABLE rides (
    id INTEGER PRIMARY KEY,
    start_location VARCHAR(12) NOT NULL,
    end_location VARCHAR(12) NOT NULL,
    rider_id INTEGER NOT NULL REFERENCES riders(id),
    driver_id INTEGER DEFAULT NULL,
    fare_estimate_pence INTEGER,
    status TEXT NOT NULL DEFAULT 'unconfirmed'
        CHECK(status IN ('unconfirmed', 'reserved', 'confirmed', 'completed')),
    FOREIGN KEY (driver_id) REFERENCES drivers(id)
);

-- Enforces at the DB level: one unconfirmed ride per rider at a time.
-- A partial index only indexes rows matching the WHERE clause, so once
-- a ride moves out of 'unconfirmed' the slot opens up again automatically.
CREATE UNIQUE INDEX one_unconfirmed_ride_per_rider
    ON rides(rider_id)
    WHERE status = 'unconfirmed';
"""

conn = sqlite3.connect(":memory:", check_same_thread=False)
conn.row_factory = sqlite3.Row
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript(SCHEMA)

print("Tables:", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()])

Tables: ['drivers', 'riders', 'rides']


## API

> After editing any cell below, re-run from **App** down through **Client**.

In [172]:
# ── App + models ──────────────────────────────────────────────────────────────
app = FastAPI(title="[Problem Name]")

class CreateRider(BaseModel):
    name: str

class CreateDriver(BaseModel):
    name: str

class CreateRide(BaseModel):
    start_location: str
    end_location: str

class UpdateDriverLocation(BaseModel):
    location: str


In [173]:
@app.post("/riders", status_code=201)
def create_rider(payload: CreateRider):
    with conn:
        cur = conn.execute("INSERT INTO riders(name) VALUES (?)", (payload.name,))
    return {"status": "ok"}

@app.post("/drivers", status_code=201)
def create_driver(payload: CreateDriver):
    with conn:
        cur = conn.execute("INSERT INTO drivers(name) VALUES (?)", (payload.name,))
    return {"status": "ok"}

In [174]:
@app.post("/rides")
def fare_estimate(payload: CreateRide, rider_id: int):
    fare_pence = abs(int(payload.end_location) - int(payload.start_location))
    try:
        with conn:
            cur = conn.execute(
                """INSERT INTO rides(start_location, end_location, rider_id, fare_estimate_pence, status)
                   VALUES (?, ?, ?, ?, 'unconfirmed')""",
                (payload.start_location, payload.end_location, rider_id, fare_pence),
            )
    except sqlite3.IntegrityError:
        # The partial unique index fires if this rider already has an unconfirmed ride.
        raise HTTPException(status_code=409, detail="Rider already has an unconfirmed ride")
    return {"status": "ok", "ride_id": cur.lastrowid, "fare_estimate_pence": fare_pence}

In [175]:
@app.patch("/drivers", status_code=201)
def update_driver_location(payload: UpdateDriverLocation, driver_id: int):
    with conn:
        cur = conn.execute(
            "UPDATE drivers SET location = ? WHERE id = ?",
            (payload.location, driver_id),
        )
    return {"status": "ok"}

In [176]:
@app.post("/reservations")
def reserve_ride(driver_id: int):
    driver = conn.execute(
        "SELECT location, reserved_ride_id FROM drivers WHERE id = ?", (driver_id,)
    ).fetchone()
    if driver is None:
        raise HTTPException(status_code=404, detail="Driver not found")
    if driver["location"] is None:
        raise HTTPException(status_code=400, detail="Driver location not set")

    # Idempotency: already reserved — return the same ride.
    if driver["reserved_ride_id"] is not None:
        ride = conn.execute(
            "SELECT id, start_location, end_location, fare_estimate_pence FROM rides WHERE id = ?",
            (driver["reserved_ride_id"],),
        ).fetchone()
        return {
            "ride_id": ride["id"],
            "start_location": ride["start_location"],
            "end_location": ride["end_location"],
            "fare_estimate_pence": ride["fare_estimate_pence"],
        }

    driver_loc = int(driver["location"])

    ride = conn.execute(
        """
        SELECT id, start_location, end_location, fare_estimate_pence
        FROM rides
        WHERE status = 'unconfirmed'
        ORDER BY ABS(CAST(start_location AS INTEGER) - ?)
        LIMIT 1
        """,
        (driver_loc,),
    ).fetchone()

    if ride is None:
        raise HTTPException(status_code=404, detail="No available rides")

    with conn:
        conn.execute("UPDATE rides SET status = 'reserved' WHERE id = ?", (ride["id"],))
        conn.execute(
            "UPDATE drivers SET reserved_ride_id = ? WHERE id = ?", (ride["id"], driver_id)
        )

    return {
        "ride_id": ride["id"],
        "start_location": ride["start_location"],
        "end_location": ride["end_location"],
        "fare_estimate_pence": ride["fare_estimate_pence"],
    }

In [177]:
@app.patch("/rides")
def accept_ride(driver_id: int):
    driver = conn.execute(
        "SELECT reserved_ride_id FROM drivers WHERE id = ?", (driver_id,)
    ).fetchone()
    if driver is None:
        raise HTTPException(status_code=404, detail="Driver not found")
    if driver["reserved_ride_id"] is None:
        raise HTTPException(status_code=400, detail="Driver has no reserved ride")

    ride_id = driver["reserved_ride_id"]

    with conn:
        conn.execute(
            "UPDATE drivers SET reserved_ride_id = NULL WHERE id = ?", (driver_id,)
        )
        conn.execute(
            "UPDATE rides SET driver_id = ?, status = 'confirmed' WHERE id = ?",
            (driver_id, ride_id),
        )

    return {"status": "ok", "ride_id": ride_id}

In [178]:
@app.delete("/reservations")
def decline_ride(driver_id: int):
    driver = conn.execute(
        "SELECT reserved_ride_id FROM drivers WHERE id = ?", (driver_id,)
    ).fetchone()
    if driver is None:
        raise HTTPException(status_code=404, detail="Driver not found")
    if driver["reserved_ride_id"] is None:
        raise HTTPException(status_code=400, detail="Driver has no reserved ride")

    ride_id = driver["reserved_ride_id"]

    # Release the reservation: ride returns to the unconfirmed pool so another
    # driver can pick it up. The partial unique index on rides(rider_id) WHERE
    # status='unconfirmed' is still satisfied because the rider only ever had
    # this one ride in flight.
    with conn:
        conn.execute(
            "UPDATE drivers SET reserved_ride_id = NULL WHERE id = ?", (driver_id,)
        )
        conn.execute(
            "UPDATE rides SET status = 'unconfirmed' WHERE id = ? AND status = 'reserved'",
            (ride_id,),
        )

    return {"status": "ok", "ride_id": ride_id}

In [179]:
# ── Client ────────────────────────────────────────────────────────────────────
client = TestClient(app, raise_server_exceptions=True)
print(client.get("/healthz").json())

{'detail': 'Not Found'}


## Helpers

In [180]:
def call(method: str, path: str, **kwargs):
    r = getattr(client, method)(path, **kwargs)
    body = r.json() if r.content else None
    print(f"{method.upper():6s} {path}  →  {r.status_code}")
    if body is not None:
        print(json.dumps(body, indent=2))
    return r


def df(table: str) -> pd.DataFrame:
    return pd.read_sql(f"SELECT * FROM {table}", conn)


def query(sql: str, *params) -> pd.DataFrame:
    return pd.read_sql(sql, conn, params=list(params) if params else None)


def show_all():
    names = [r[0] for r in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()]
    for name in names:
        count = conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
        print(f"\n── {name} ({count} rows) ──")
        display(pd.read_sql(f"SELECT * FROM {name}", conn))

## Demo

In [181]:
call("post", "/riders", json={"name": "Angelo"})
call("post", "/riders", json={"name": "Vincent"})
df("riders")

POST   /riders  →  201
{
  "status": "ok"
}
POST   /riders  →  201
{
  "status": "ok"
}


,id,name
0,1,Angelo
1,2,Vincent


In [182]:
call("post", "/drivers", json={"name": "Michael"})
df("drivers")

POST   /drivers  →  201
{
  "status": "ok"
}


,id,name,location,reserved_ride_id
0,1,Michael,None,None


In [183]:
call("post", "/rides", json={"start_location": "5", "end_location": "9"}, params={"rider_id": 1})
call("post", "/rides", json={"start_location": "7", "end_location": "12"}, params={"rider_id": 2})

df("rides")

POST   /rides  →  200
{
  "status": "ok",
  "ride_id": 1,
  "fare_estimate_pence": 4
}
POST   /rides  →  200
{
  "status": "ok",
  "ride_id": 2,
  "fare_estimate_pence": 5
}


,id,start_location,end_location,rider_id,driver_id,fare_estimate_pence,status
0,1,5,9,1,None,4,unconfirmed
1,2,7,12,2,None,5,unconfirmed


Second ride for rider 1 should be rejected — they already have an unconfirmed ride

In [184]:
call("post", "/rides", json={"start_location": "5", "end_location": "9"}, params={"rider_id": 1})

POST   /rides  →  409
{
  "detail": "Rider already has an unconfirmed ride"
}


<Response [409 Conflict]>

In [185]:
call("patch", "/drivers", json={"location": "0"}, params={"driver_id": 1})
df("drivers")

PATCH  /drivers  →  201
{
  "status": "ok"
}


,id,name,location,reserved_ride_id
0,1,Michael,0,None


In [186]:
call("patch", "/drivers", json={"location": "1"}, params={"driver_id": 1})
df("drivers")

PATCH  /drivers  →  201
{
  "status": "ok"
}


,id,name,location,reserved_ride_id
0,1,Michael,1,None


In [187]:
call("post", "/reservations", params={"driver_id": 1})
df("rides")


POST   /reservations  →  200
{
  "ride_id": 1,
  "start_location": "5",
  "end_location": "9",
  "fare_estimate_pence": 4
}


,id,start_location,end_location,rider_id,driver_id,fare_estimate_pence,status
0,1,5,9,1,None,4,reserved
1,2,7,12,2,None,5,unconfirmed


Retry — should return the same ride (idempotent), not grab a second one.

In [188]:
call("post", "/reservations", params={"driver_id": 1})

POST   /reservations  →  200
{
  "ride_id": 1,
  "start_location": "5",
  "end_location": "9",
  "fare_estimate_pence": 4
}


<Response [200 OK]>

Driver declines this ride, moves closer to the start location of the OTHER ride. Calling the `/reservations` endpoint now reserves THAT ride.

In [189]:
call("delete", "/reservations", params={"driver_id": 1})
call("patch", "/drivers", json={"location": "7"}, params={"driver_id": 1})
call("post", "/reservations", params={"driver_id": 1})
df("rides")

DELETE /reservations  →  200
{
  "status": "ok",
  "ride_id": 1
}
PATCH  /drivers  →  201
{
  "status": "ok"
}
POST   /reservations  →  200
{
  "ride_id": 2,
  "start_location": "7",
  "end_location": "12",
  "fare_estimate_pence": 5
}


,id,start_location,end_location,rider_id,driver_id,fare_estimate_pence,status
0,1,5,9,1,None,4,unconfirmed
1,2,7,12,2,None,5,reserved


Driver accepts that ride

In [190]:
call("patch", "/rides", params={"driver_id": 1})
df("rides")

PATCH  /rides  →  200
{
  "status": "ok",
  "ride_id": 2
}


,id,start_location,end_location,rider_id,driver_id,fare_estimate_pence,status
0,1,5,9,1,NaN,4,unconfirmed
1,2,7,12,2,1.0,5,confirmed


## All tables

In [191]:
show_all()


── drivers (1 rows) ──


,id,name,location,reserved_ride_id
0,1,Michael,7,None



── riders (2 rows) ──


,id,name
0,1,Angelo
1,2,Vincent



── rides (2 rows) ──


,id,start_location,end_location,rider_id,driver_id,fare_estimate_pence,status
0,1,5,9,1,NaN,4,unconfirmed
1,2,7,12,2,1.0,5,confirmed
